# Clase 11 — Tablero de ajedrez

Vamos a construir un tablero que sepa pintarse, y un peón que sepa calcular sus movimientos. La clave de hoy es cómo dos objetos se comunican entre sí usando atributos y métodos.

---

## Diseño antes de escribir código

Antes de escribir una línea, definimos qué sabe cada clase y qué no:

| | `Tablero` | `Peon` |
|---|---|---|
| Sabe | Dónde está cada pieza | Su color y si ya se movió |
| Sabe | Qué celdas pintar verde | Cómo calcular sus movimientos |
| No sabe | Cómo se mueve cada pieza | Nada del tablero (solo pregunta) |

El peón **pregunta** al tablero si una celda está libre. El tablero **no sabe** de reglas de ajedrez.

---

## Clase `Tablero`

In [1]:
class Tablero:
    _BLANCA = "\033[47m"   # fondo blanco
    _NEGRA  = "\033[100m"  # fondo gris oscuro
    _VERDE  = "\033[42m"   # fondo verde (movimiento posible)
    _RESET  = "\033[0m"

    def __init__(self):
        self._grid     = [[None] * 8 for _ in range(8)]  # None = celda vacía
        self._marcadas = set()                            # (fila, col) a pintar verde

    # --- interfaz pública para las piezas ---

    def celda_libre(self, fila, col):
        """Retorna True si la celda no tiene ninguna pieza."""
        return self._grid[fila][col] is None

    def hay_enemigo(self, fila, col, color_amigo):
        """Retorna True si la celda tiene una pieza de color distinto."""
        pieza = self._grid[fila][col]
        return pieza is not None and pieza.color != color_amigo

    def colocar(self, pieza, fila, col):
        """Pone una pieza en la celda y le informa su posición."""
        self._grid[fila][col] = pieza
        pieza._fila = fila
        pieza._col  = col

    def mostrar_movimientos(self, pieza):
        """Pide a la pieza sus movimientos, los marca y muestra el tablero."""
        self._marcadas = set(pieza.movimientos_posibles(self))
        self.imprimir()
        self._marcadas = set()

    # --- impresión ---

    def _color_casilla(self, fila, col):
        """Retorna el código de color de fondo según el color de la casilla."""
        if (fila + col) % 2 == 0:
            return self._BLANCA
        return self._NEGRA

    def imprimir(self):
        print("\n    a   b   c   d   e   f   g   h")
        print("  +" + "---+" * 8)
        for fila in range(8):
            print(f"{8 - fila} |", end="")
            for col in range(8):
                pieza   = self._grid[fila][col]
                simbolo = str(pieza) if pieza else " "
                fondo   = self._VERDE if (fila, col) in self._marcadas else self._color_casilla(fila, col)
                print(f"{fondo} {simbolo} {self._RESET}|", end="")
            print(f" {8 - fila}")
        print("  +" + "---+" * 8)
        print("    a   b   c   d   e   f   g   h\n")

---

## Clase `Peon`

El peón conoce las reglas de su propio movimiento:
- Avanza 1 casilla hacia adelante si está libre
- Avanza 2 casillas si aún no se ha movido y ambas están libres
- Captura en diagonal si hay un enemigo

Para saber si una celda está libre o tiene un enemigo, **le pregunta al tablero** — no accede a `_grid` directamente.

In [2]:
class Peon:
    def __init__(self, color):
        self.color   = color    # "blanco" o "negro"
        self._fila   = None     # asignado por tablero.colocar()
        self._col    = None
        self._movido = False    # controla el avance doble inicial

    def __str__(self):
        return "♙" if self.color == "blanco" else "♟"

    def movimientos_posibles(self, tablero):
        """Retorna lista de (fila, col) a los que puede moverse."""
        movs      = []
        direccion = -1 if self.color == "blanco" else 1   # blanco sube, negro baja
        f, c      = self._fila, self._col
        sig_fila  = f + direccion

        # avance simple
        if 0 <= sig_fila < 8 and tablero.celda_libre(sig_fila, c):
            movs.append((sig_fila, c))

            # avance doble desde posición inicial
            if not self._movido:
                doble = f + 2 * direccion
                if 0 <= doble < 8 and tablero.celda_libre(doble, c):
                    movs.append((doble, c))

        # capturas en diagonal
        for dc in [-1, 1]:
            nc = c + dc
            if 0 <= sig_fila < 8 and 0 <= nc < 8:
                if tablero.hay_enemigo(sig_fila, nc, self.color):
                    movs.append((sig_fila, nc))

        return movs

---

## Demo 1 — Peón en posición inicial

Un peón blanco recién puesto. Debe poder avanzar 1 o 2 casillas.

In [3]:
tablero = Tablero()
peon    = Peon("blanco")

tablero.colocar(peon, fila=6, col=3)   # fila 6 = fila 2 del tablero (posición inicial blanco)

print("Tablero inicial:")
tablero.imprimir()

print("Movimientos posibles del peón:")
tablero.mostrar_movimientos(peon)

Tablero inicial:

    a   b   c   d   e   f   g   h
  +---+---+---+---+---+---+---+---+
8 |   |   |   |   |   |   |   |   | 8
7 |   |   |   |   |   |   |   |   | 7
6 |   |   |   |   |   |   |   |   | 6
5 |   |   |   |   |   |   |   |   | 5
4 |   |   |   |   |   |   |   |   | 4
3 |   |   |   |   |   |   |   |   | 3
2 |   |   |   | ♙ |   |   |   |   | 2
1 |   |   |   |   |   |   |   |   | 1
  +---+---+---+---+---+---+---+---+
    a   b   c   d   e   f   g   h

Movimientos posibles del peón:

    a   b   c   d   e   f   g   h
  +---+---+---+---+---+---+---+---+
8 |   |   |   |   |   |   |   |   | 8
7 |   |   |   |   |   |   |   |   | 7
6 |   |   |   |   |   |   |   |   | 6
5 |   |   |   |   |   |   |   |   | 5
4 |   |   |   |   |   |   |   |   | 4
3 |   |   |   |   |   |   |   |   | 3
2 |   |   |   | ♙ |   |   |   |   | 2
1 |   |   |   |   |   |   |   |   | 1
  +---+---+---+---+---+---+---+---+
    a   b   c   d   e   f   g   h



---

## Demo 2 — Enemigo bloqueando y en diagonal

Un peón negro bloquea el avance directo. Otro está en diagonal — el blanco puede capturarlo.

In [4]:
tablero = Tablero()

blanco  = Peon("blanco")
negro1  = Peon("negro")   # bloquea el avance
negro2  = Peon("negro")   # está en diagonal — capturable

tablero.colocar(blanco, fila=4, col=3)
tablero.colocar(negro1, fila=3, col=3)  # justo adelante
tablero.colocar(negro2, fila=3, col=4)  # diagonal derecha

print("Movimientos del peón blanco (avance bloqueado, captura disponible):")
tablero.mostrar_movimientos(blanco)

Movimientos del peón blanco (avance bloqueado, captura disponible):

    a   b   c   d   e   f   g   h
  +---+---+---+---+---+---+---+---+
8 |   |   |   |   |   |   |   |   | 8
7 |   |   |   |   |   |   |   |   | 7
6 |   |   |   |   |   |   |   |   | 6
5 |   |   |   | ♟ | ♟ |   |   |   | 5
4 |   |   |   | ♙ |   |   |   |   | 4
3 |   |   |   |   |   |   |   |   | 3
2 |   |   |   |   |   |   |   |   | 2
1 |   |   |   |   |   |   |   |   | 1
  +---+---+---+---+---+---+---+---+
    a   b   c   d   e   f   g   h



---

## Para discutir

Mirando el código:

1. `Peon.movimientos_posibles()` llama a `tablero.celda_libre()` y `tablero.hay_enemigo()`. ¿Por qué no accede a `tablero._grid` directamente?

2. `tablero.colocar()` escribe en `pieza._fila` y `pieza._col`. ¿Por qué el tablero tiene esa responsabilidad y no el propio peón?

3. ¿Qué habría que cambiar para agregar una segunda pieza, por ejemplo un `Alfil`? ¿Hay que modificar `Tablero`?

---

## Ejercicio

Agrega la clase `Torre`. Una torre se mueve en línea recta (horizontal o vertical) cualquier número de casillas, pero **no puede saltar piezas**.

- Implementa `movimientos_posibles(tablero)` siguiendo el mismo patrón que `Peon`
- Solo usa `tablero.celda_libre()` y `tablero.hay_enemigo()` — no accedas a `_grid`
- Pruébala en el tablero con algunas piezas bloqueando el camino

In [ ]:
# Tu clase Torre aquí
